In [2]:
import re
import pandas as pd
import numpy as np


In [4]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion")

# pd.DataFrame(dataset["train"]).to_csv("train.csv", index=False)
# pd.DataFrame(dataset["validation"]).to_csv("val.csv", index=False)
# pd.DataFrame(dataset["test"]).to_csv("test.csv", index=False)

# print("Saved successfully")

In [5]:
print(dataset["train"].features)

{'text': Value('string'), 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}


In [63]:
df = pd.read_csv("train.csv")
df.head()

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


In [7]:
# removing url
def remove_url(text):
    text = re.sub(r"https?://\S+","",text)
    return text

In [8]:
# removing punctuations
def remove_punc(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text

In [9]:
# removing html tags
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text

In [10]:
# removing stop words
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words = stopwords.words("english")
    for i in tokens:
        if i in stop_words:
            text = text.replace(i,"")

    return text    

In [11]:
# stemming
from nltk.stem import PorterStemmer

def stemming(text):
    ps=PorterStemmer()
    stem_words=[]
    tokens = word_tokenize(text)
    for words in tokens:
        stemmed_tokens = ps.stem(words)
        stem_words.append(stemmed_tokens)

    return " ".join(stem_words)    

In [64]:
def preprocess(text):
    text = text.lower()
    text = remove_url(text)
    text = remove_html(text)
    text = remove_punc(text)
    text = remove_stopwords(text)
    text = stemming(text)

    return text

df["text"] = df["text"].apply(preprocess)

In [73]:
X=df["text"]
y=df["label"]

In [74]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [75]:
# vectorization

from sklearn.feature_extraction.text import  TfidfVectorizer

tf=TfidfVectorizer()

X_train = tf.fit_transform(X_train)
X_test = tf.transform(X_test)

In [69]:
X_train.shape

(12800, 15154)

In [76]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [77]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [78]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [79]:
train_loader = DataLoader(train_set,shuffle=True,batch_size=64)
test_loader = DataLoader(test_set,shuffle=True,batch_size=64)

In [80]:
# creating RNN
import torch.nn as nn
import torch.optim as optim

In [100]:
# many to many architecture

class RNN(nn.Module):
    def __init__(self, input_size, hidden_layer_size=500, num_layers=2,output_size=6):
        super().__init__()

        self.hidden_layer_size = hidden_layer_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_layer_size,
            num_layers=num_layers,
            batch_first=True

        )
        self.dropout = nn.Dropout(0.3)

        # Fully Connected Layer
        self.fc = nn.Linear(hidden_layer_size,output_size)

    def forward(self, x):
        # Initial hidden state
        h0 = torch.zeros(
            self.num_layers,
            x.size(0),
            self.hidden_layer_size
        ).to(x.device)

        # Forward pass through RNN
        out, _ = self.rnn(x, h0)

        # Take only the last timestep output
        out = out[:, -1, :]
        
        # dropout
        out = self.dropout(out)    #works only in training and automatically turns off in eval mode

        # Pass through fully connected layer
        out = self.fc(out)

        return out    

In [82]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [101]:
input_size=X_train.shape[1]
model = RNN(input_size).to(device)

criterion = nn.CrossEntropyLoss()   #No softmax needed manually because CrossEntropyLoss applies it internally.
optimizer=optim.Adam(model.parameters(),lr=0.0005)

In [102]:
# Training the RNN
epochs=20

for epoch in range(epochs):
    model.train()

    for x,y in train_loader:
        x = x.to(device)
        y = y.to(device).long()
        
        optimizer.zero_grad()

        x=x.unsqueeze(1)

        output=model(x)
        loss=criterion(output,y)
        loss.backward()
        optimizer.step()

    print(f"epoch={epoch}/{epochs} and the loss = {loss.item()}")    

epoch=0/20 and the loss = 0.6336873769760132
epoch=1/20 and the loss = 0.6108733415603638
epoch=2/20 and the loss = 0.04840141534805298
epoch=3/20 and the loss = 0.014029759913682938
epoch=4/20 and the loss = 0.0328768827021122
epoch=5/20 and the loss = 0.03669913113117218
epoch=6/20 and the loss = 0.0039641959592700005
epoch=7/20 and the loss = 0.008202750235795975
epoch=8/20 and the loss = 0.0033737325575202703
epoch=9/20 and the loss = 0.0013586929999291897
epoch=10/20 and the loss = 0.0012474488466978073
epoch=11/20 and the loss = 0.06946464627981186
epoch=12/20 and the loss = 0.033159613609313965
epoch=13/20 and the loss = 0.0018002327997237444
epoch=14/20 and the loss = 0.0020737056620419025
epoch=15/20 and the loss = 0.0015596391167491674
epoch=16/20 and the loss = 0.033789604902267456
epoch=17/20 and the loss = 0.13280817866325378
epoch=18/20 and the loss = 0.0013996048364788294
epoch=19/20 and the loss = 0.005546091124415398


In [ ]:
# testing

In [103]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        x = x.unsqueeze(1)

        outputs = model(x)

        _, predicted = torch.max(outputs, 1)

        total += y.size(0)
        correct += (predicted == y).sum().item()

print("Test Accuracy:", 100 * correct / total)

Test Accuracy: 77.46875


In [120]:
torch.save(model.state_dict(), "emotion_model.pth")
# this saves:

# learned weights
# biases
# trained parameters

# File created:

# emotion_model.pth

In [121]:
# This saves your trained TF-IDF object.
import joblib
joblib.dump(tf, "tfidf.pkl")

['tfidf.pkl']

In [ ]:
# testing on test.csv

In [104]:
# load test file
test_df = pd.read_csv("test.csv")

# split features and labels
x_test2 = test_df["text"]
y_test2 = test_df["label"]

# preprocessing
x_test2 = x_test2.apply(preprocess)

# transform using trained tfidf
x_test2 = tf.transform(x_test2)

# convert into tensors
x_test2 = torch.tensor(x_test2.toarray(), dtype=torch.float32)
y_test2 = torch.tensor(y_test2.values, dtype=torch.long)

# dataloader
test2_dataset = TensorDataset(x_test2, y_test2)
test2_loader = DataLoader(test2_dataset, batch_size=32, shuffle=False)


In [105]:
# testing
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for x, y in test2_loader:
        x = x.to(device)
        y = y.to(device)

        x = x.unsqueeze(1)

        outputs = model(x)

        _, predicted = torch.max(outputs, 1)

        total += y.size(0)
        correct += (predicted == y).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 75.55%


In [ ]:
# manual test

In [106]:
label_map = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

def predict_emotion(sentence):
    model.eval()

    # preprocess
    sentence = preprocess(sentence)

    # tfidf transform
    vector = tf.transform([sentence])

    # convert to tensor
    vector = torch.tensor(vector.toarray(), dtype=torch.float32).to(device)

    # add sequence dimension
    vector = vector.unsqueeze(1)

    # prediction
    with torch.no_grad():
        output = model(vector)

        predicted_class = torch.argmax(output, dim=1).item()

    return label_map[predicted_class]

In [ ]:
sentence = ""
print(predict_emotion(sentence))

anger
